<a href="https://colab.research.google.com/github/pikey-msc/RiesgosFinancieros/blob/master/2027-1/Bonos_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/pikey-msc/RiesgosFinancieros/blob/master/2026-1/Bonos_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Valuación de Instrumentos Gubernamentales de Deuda

**Riesgos Financieros — 2026-1**

Este cuaderno es una introducción práctica a la **valuación de deuda gubernamental mexicana** usando curvas de tasas de interés. Vamos a valuar tres instrumentos muy comunes en el mercado local:

| Instrumento | ¿Qué es? | ¿Cómo paga? |
|---|---|---|
| **CETES** (bono cupón cero) | Título de deuda de corto plazo | Un solo pago al vencimiento (el valor nominal). No paga intereses periódicos. |
| **Bono M** | Bono de tasa fija a mediano/largo plazo | Cupones fijos, normalmente semestrales, más el nominal al vencimiento. |
| **Bonde D** | Bono de tasa flotante (revisable) | Cupones cada 28 días, referenciados a la tasa de fondeo bancario. |

### 🗺️ Ruta del cuaderno
1. **Marco teórico** — curvas cupón cero, curvas *yield* e interpolación de tasas.
2. **Parámetros** — cambia la fecha de valuación, el método de interpolación y las características de cada bono usando los **campos de formulario** (el panel a la derecha de cada celda, o el ícono ⋮ → *Mostrar código* para ver el Python).
3. **Carga de datos** — se descargan curvas de mercado históricas del repositorio del curso.
4. **Visualización** — cómo luce la curva de tasas el día de valuación.
5. **Valuación** — CETES, Bonos M (dos métodos) y Bondes D, paso a paso, con la fórmula matemática antes de cada bloque de código.
6. **Resumen** — tabla y gráfica comparativa de resultados.

> 💡 **Tip:** en Colab, las celdas con formulario muestran solo los campos de entrada; el código detrás sigue ahí, solo está colapsado. Usa el menú ⋮ de la celda (o *Ver → Formularios → Mostrar/ocultar código*) para revisarlo o editarlo directamente.

**Cómo usarlo:** la primera vez, ejecuta todo con *Entorno de ejecución → Ejecutar todas*. Después, cambia cualquier campo de formulario y vuelve a ejecutar el cuaderno desde esa celda hacia abajo para ver cómo cambian los resultados.


In [ ]:
# @title 📦 Clonar el repositorio del curso (datos e insumos) { display-mode: "form" }
# Descarga la versión más reciente del repositorio a la máquina virtual de Colab.
# Ahí viven los archivos .txt con las curvas de tasas históricas que usaremos
# para interpolar y valuar los bonos.
!rm -rf RiesgosFinancieros
!git clone "https://github.com/pikey-msc/RiesgosFinancieros/"


Cloning into 'RiesgosFinancieros'...
remote: Enumerating objects: 1338, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 1338 (delta 102), reused 68 (delta 68), pack-reused 1207 (from 2)
Receiving objects: 100% (1338/1338), 72.80 MiB | 9.52 MiB/s, done.
Resolving deltas: 100% (836/836), done.


## 🧮 Marco teórico: curvas de tasas de interés

Para valuar cualquier instrumento de deuda necesitamos **traer a valor presente** los flujos futuros que promete pagar. Para eso usamos una **curva de tasas de interés**, que nos dice cuál es la tasa adecuada para descontar un flujo que ocurre dentro de $p$ días.

En este cuaderno usamos dos tipos de curva, construidas con datos de mercado:

- **Curva cupón cero** (`tasa_guber.txt`): tasas de descuento "puras". Se usa para descontar flujo por flujo (método de cupón cero).
- **Curva *yield*** (`tasa_yield.txt`): la tasa interna de retorno (TIR) de mercado de un bono cuponado con esas características. Se usa para valuar de un solo golpe con la fórmula de anualidad de un bono.

### El problema de la interpolación

Las curvas de mercado solo traen tasas para ciertos **plazos fijos** (nodos: 1, 7, 30, 90, 180, 270, 360... días), pero nuestros bonos vencen o pagan cupones en plazos que caen **entre** esos nodos. Hay que **interpolar**. El cuaderno ofrece dos métodos, elegibles con un campo de formulario:

1. **Interpolación lineal** — traza una línea recta entre dos nodos consecutivos y lee el valor sobre esa recta. Simple, pero no es consistente con cómo se componen las tasas en el tiempo.
2. **Tasa alambrada** *(la que usa el mercado mexicano)* — en vez de interpolar la tasa directamente, se interpola de forma que las **tasas forward implícitas** entre nodos sean consistentes. Si $t_c$ y $t_l$ son las tasas de los nodos que rodean al plazo $p$ (entre $p_i$ y $p_{i+1}$), la tasa interpolada resuelve:

$$ t_{p} = \left[ \left( \frac{1+t_l \cdot p_{i+1}/360}{1+t_c \cdot p_i/360} \right)^{\frac{p-p_i}{p_{i+1}-p_i}} \cdot \left(1+t_c \cdot \frac{p_i}{360}\right) - 1\right] \cdot \frac{360}{p}$$

Esto es exactamente lo que hace la función `talamb()` de abajo. Fuera del rango de nodos, ambos métodos simplemente sostienen el valor del nodo más cercano (no extrapolan).

> 🔎 **Para explorar:** más adelante podrás cambiar el método de interpolación con un campo de formulario y comparar cómo cambian los resultados de valuación.


# 🔧 Funciones auxiliares

In [ ]:
# @title Función talamb(): interpolación por "tasa alambrada" { display-mode: "both" }
import numpy as np

def talamb(nodos, curva, plazos):
    """
    Interpola tasas de interés de forma consistente con las tasas forward
    implícitas entre nodos (método "tasa alambrada", estándar del mercado
    mexicano para curvas gubernamentales).

    Parámetros
    ----------
    nodos  : plazos (en días) donde conocemos la tasa de la curva.
    curva  : tasas de interés observadas en cada nodo.
    plazos : plazos (en días) donde queremos conocer la tasa interpolada
             (por ejemplo, los vencimientos/cupones de nuestros bonos).

    Regresa
    -------
    Un arreglo con la tasa interpolada para cada elemento de `plazos`.
    Fuera del rango de `nodos` se sostiene el valor del nodo más cercano
    (no se extrapola).
    """
    nodos = nodos.flatten()
    curva = curva.flatten()
    plazos = plazos.flatten()

    # Aseguramos que los nodos estén ordenados para np.interp
    sorted_indices = np.argsort(nodos)
    nodos_sorted = nodos[sorted_indices]
    curva_sorted = curva[sorted_indices]

    # Índices del nodo inferior para cada plazo a interpolar
    indices = np.searchsorted(nodos_sorted, plazos) - 1
    indices[indices < 0] = 0
    indices[indices >= len(nodos_sorted) - 1] = len(nodos_sorted) - 2

    nodos_i = nodos_sorted[indices]
    nodos_i_plus_1 = nodos_sorted[indices + 1]
    TC_interp = curva_sorted[indices]
    TL_interp = curva_sorted[indices + 1]

    # Tasa forward-consistente entre los dos nodos que rodean al plazo
    TF = np.where(plazos < nodos_sorted[0], curva_sorted[0],
             np.where(plazos > nodos_sorted[-1], curva_sorted[-1],
                      ((((1 + TL_interp * nodos_i_plus_1 / 360) / (1 + TC_interp * nodos_i / 360)) **
                        ((plazos - nodos_i) / (nodos_i_plus_1 - nodos_i)) *
                        (1 + TC_interp * nodos_i / 360)) - 1) * 360 / plazos))

    return TF


# 🎛️ Parámetros de valuación

Aquí definimos **todo lo necesario para valuar cada instrumento**: la fecha de valuación, el método de interpolación, y las características de cada bono (plazos, tasas cupón, número de contratos y valor nominal).

Cada bloque está en una **celda de formulario** 📋 — no necesitas tocar código para experimentar, solo cambia los valores en el panel de la derecha y vuelve a ejecutar el cuaderno desde ahí hacia abajo. (Para ver/editar el Python directamente: menú ⋮ de la celda → *Mostrar código*.)

Cada bono se define como un **vector de 5 posiciones de ejemplo**: puedes pensarlo como una pequeña cartera con 5 títulos del mismo tipo pero distinto plazo, algunas posiciones "largas" (compradas) y otras "cortas" (vendidas, con signo negativo).


In [ ]:
# @title ⚙️ Parámetros generales { display-mode: "form" }
import numpy as np
from datetime import datetime, timedelta

fecha_valuacion = '2023-03-10'  # @param {type:"date"}
fval = datetime.strptime(fecha_valuacion.replace('-', ''), "%Y%m%d")  # Fecha de valuación

metodo_interpolacion = "Tasa alambrada"  # @param ["Interpolaci\u00F3n lineal", "Tasa alambrada"]
# La "tasa alambrada" es el método usado en el mercado mexicano (ver Marco Te\u00f3rico):
# interpola de forma consistente con las tasas forward impl\u00edcitas entre nodos.
itpl = 0 if metodo_interpolacion == "Interpolaci\u00F3n lineal" else 1

print(f"Fecha de valuaci\u00f3n elegida: {fval:%d/%m/%Y}")
print(f"M\u00e9todo de interpolaci\u00f3n: {metodo_interpolacion} (itpl = {itpl})")


In [ ]:
# @title 🧾 Parámetros — CETES (bono cupón cero) { display-mode: "form" }
# Archivo con la curva cupón cero histórica (nodos en columnas, fechas en filas)
base = "RiesgosFinancieros/2022-1/Insumos/tasa_guber.txt"

plazos_bcc = [37, 40, 55, 120, 180]  # @param {type:"raw"}
contratos_bcc = [22000, -29000, 29000, -46000, 10000]  # @param {type:"raw"}
nominal_bcc = 10  # @param {type:"number"}

plazos_bcc = np.array(plazos_bcc)        # Plazo (d\u00edas) al vencimiento de cada t\u00edtulo
contratos_bcc = np.array(contratos_bcc)  # N\u00famero de contratos (posici\u00f3n; negativo = corto)


In [ ]:
# @title 🧾 Parámetros — Bono M (tasa fija) { display-mode: "form" }
# Curva cupón cero y curva yield históricas para valuar el Bono M
btasadesc_bm = "RiesgosFinancieros/2022-1/Insumos/tasa_guber.txt"
btasayield_bm = "RiesgosFinancieros/2022-1/Insumos/tasa_yield.txt"

tfcupon_bm = [0.065, 0.0675, 0.07, 0.075, 0.078]  # @param {type:"raw"}
plazos_bm = [378, 405, 550, 1200, 1800]  # @param {type:"raw"}
plazocupon_bm = [182, 182, 182, 182, 182]  # @param {type:"raw"}
contratos_bm = [22000, -29000, 29000, -46000, 10000]  # @param {type:"raw"}
nominal_bm = 100  # @param {type:"number"}

tfcupon_bm = np.array(tfcupon_bm)        # Tasa cup\u00f3n fija de cada bono
plazos_bm = np.array(plazos_bm)          # Plazo (d\u00edas) al vencimiento
plazocupon_bm = np.array(plazocupon_bm)  # Periodicidad del cup\u00f3n (d\u00edas)
contratos_bm = np.array(contratos_bm)    # N\u00famero de contratos (posici\u00f3n)


In [ ]:
# @title 🧾 Parámetros — Bonde D (tasa flotante) { display-mode: "form" }
# Curva de sobretasa/spread y serie histórica de la tasa de fondeo bancario
btasadescst = "RiesgosFinancieros/2024-1/Tarea/tasa_guber_st.txt"
btasafondeo = "RiesgosFinancieros/2024-1/Tarea/tfondeo.txt"

plazos_bdm = [358, 405, 550, 1200, 1800]  # @param {type:"raw"}
plazocupon_bdm = [28, 28, 28, 28, 28]  # @param {type:"raw"}
contratos_bdm = [220, -290, 290, -460, 100]  # @param {type:"raw"}
nominal_bdm = 100  # @param {type:"number"}

plazos_bdm = np.array(plazos_bdm)
plazocupon_bdm = np.array(plazocupon_bdm)
contratos_bdm = np.array(contratos_bdm)


# 📥 Carga de datos

Con los parámetros ya definidos, toca **leer las curvas de tasas** desde los archivos del repositorio y dejarlas listas para interpolar:

- `tasa_guber.txt` → curva cupón cero (CETES y Bono M).
- `tasa_yield.txt` → curva yield (Bono M, método alterno).
- `tasa_guber_st.txt` → curva de sobretasa/spread (Bonde D).
- `tfondeo.txt` → serie diaria de la tasa de fondeo bancario (Bonde D), con la que se calcula el cupón realmente devengado en el periodo en curso.

Esta celda es principalmente "plomería" de datos (lectura de archivos, `merge` de fechas, etc.), por eso está colapsada como formulario — pero puedes expandirla si quieres ver el detalle.


In [ ]:
# @title 📥 Carga y preparación de curvas de mercado { display-mode: "form" }
import pandas as pd

# ------------------------- BONOS M (curva cup\u00f3n cero) -------------------------
data = pd.read_csv(base, sep="\t", header=None)

# n = n\u00famero de fechas disponibles; m_gov = n\u00famero de nodos + 1 (columna DATE)
n, m_gov = data.shape

# DataFrame con las tasas (sin la fila/columna de encabezado) + columna de fecha
x_orig = pd.DataFrame(data.values[1:, 1:len(data.values[0])], dtype=float)
x_orig['Date'] = pd.to_datetime(data.values[1:, 0], format='%Y%m%d')

# Nodos (plazos en d\u00edas) de la curva
nodos_gov = pd.DataFrame(data.values[0, 1:(len(data.values[0]))], dtype=int)

# ------------------------------- CURVA YIELD --------------------------------
data_yd = pd.read_csv(btasayield_bm, sep="\t", header=None)
n_yd, m_gov_yd = data_yd.shape

x_orig_yd = pd.DataFrame(data_yd.values[1:, 1:len(data_yd.values[0])], dtype=float)
x_orig_yd['Date'] = pd.to_datetime(data_yd.values[1:, 0], format='%Y%m%d')

nodos_gov_yd = pd.DataFrame(data_yd.values[0, 1:(len(data.values[0]))], dtype=int)

# --------------------------------- BONDE D ----------------------------------
# Volvemos a leer el archivo base (misma curva cup\u00f3n cero) con otro parser,
# conservando la fila/columna de encabezado para separar nodos y fechas.
data1 = pd.read_table(base, sep="\t", header=None)
n = data1.shape[0]
m_bd = data1.shape[1]
X_orig = data1.iloc[1:n, 0:m_bd]
X1_orig = pd.DataFrame(data1.iloc[1:n, 1:m_bd], dtype=float)
X1_orig['Date'] = pd.to_datetime(X_orig[0], format='%Y%m%d')
nodos = data1.iloc[0, 1:m_bd]
n -= 1

# Curva de sobretasa/spread
data3 = pd.read_table(btasadescst, sep="\t", header=None)
n3 = data3.shape[0]
m3_bd = data3.shape[1]
X3a_orig_bd = data3.iloc[1:n3, 0:m3_bd]
X3_orig_bd = pd.DataFrame(data3.iloc[1:n3, 1:m3_bd], dtype=float)
X3_orig_bd['Date'] = pd.to_datetime(X3a_orig_bd[0], format='%Y%m%d')
nodos3_bd = data3.iloc[0, 1:m3_bd]
n3 -= 1

# Serie hist\u00f3rica de tasa de fondeo
data2 = pd.read_table(btasafondeo, sep="\t", header=None)
n2 = data2.shape[0]
X2_orig = data2.iloc[1:n2, 0:2]
X2_orig_bd = X2_orig.copy()
X2_orig_bd[0] = pd.to_datetime(X2_orig_bd[0], format='%Y%m%d')
X2_orig_bd[1] = X2_orig_bd[1].astype(float)

# Rellenamos d\u00edas sin dato (fines de semana/feriados) con la \u00faltima tasa vigente
tfh = pd.date_range(start=min(X2_orig_bd[0]), end=max(X2_orig_bd[0]), freq='D')
tfhd = pd.DataFrame({'fecha': tfh}).sort_values(by='fecha', ascending=True)
X2_orig_bd = X2_orig_bd.sort_values(by=0, ascending=True).rename(columns={0: 'fecha'})
X2_orig_bd = pd.merge_asof(tfhd, X2_orig_bd, on='fecha', direction='backward')
X2_orig_bd = pd.merge(tfhd, X2_orig_bd, on='fecha', how='outer')

X1_orig = X1_orig.rename(columns={0: "fecha"})

# Tasa de fondeo vigente en la fecha de valuaci\u00f3n, y la serie de tasas
# realizadas durante el periodo de cup\u00f3n en curso (para el cup\u00f3n devengado)
tf_act = X2_orig_bd.loc[X2_orig_bd['fecha'] == fval, 1].values[0] / 100
tf_int = X2_orig_bd.loc[(X2_orig_bd['fecha'] <= fval) & (X2_orig_bd['fecha'] >= (fval - timedelta(int(plazocupon_bdm[0])))), 1] / 100
tf_int = tf_int[::-1]  # de m\u00e1s reciente a m\u00e1s antigua

X1_orig = X1_orig.sort_values('Date', ascending=True)
X1_orig['join_date'] = X1_orig['Date']
X2_orig_bd = X2_orig_bd.sort_values('fecha', ascending=True)
X2_pr = pd.merge_asof(X1_orig, X2_orig_bd, left_on='Date', right_on='fecha')

print(f"Filas disponibles en la curva cup\u00f3n cero (n): {n}")
print(f"Tasa de fondeo vigente en {fval:%d/%m/%Y}: {tf_act:.4%}")


## 👀 Visualicemos la curva de tasas

Antes de valuar nada, vale la pena **ver la curva** que vamos a usar y confirmar que la interpolación hace lo que esperamos: cada "✕" roja debe caer sobre (o muy cerca de) la línea azul de mercado, en el plazo exacto de nuestros bonos.

> ⚠️ **Nota importante:** el archivo `tasa_guber.txt` que trae la curva cupón cero solo tiene historia hasta cierta fecha. La celda de "Carga de datos" siempre toma la **fila 0** de ese archivo (la fecha más reciente disponible), que puede no coincidir exactamente con la `fecha_valuacion` que elegiste arriba — esa fecha sí se usa tal cual para buscar la tasa de fondeo de Bondes D, donde el archivo `tfondeo.txt` llega hasta 2023. Tenlo en mente al interpretar resultados.


In [ ]:
# @title 📈 Curva de tasas vigente y plazos interpolados { display-mode: "both" }
import matplotlib.pyplot as plt

nodos_plot = np.array(nodos_gov[0], dtype=float)
tasas_plot = np.array(x_orig.drop('Date', axis=1).iloc[0], dtype=float)
fecha_curva = x_orig['Date'].iloc[0]

plazos_interp = np.sort(np.unique(np.concatenate([plazos_bcc, plazos_bm])))
if itpl == 1:
    tasas_interp = talamb(nodos_plot, tasas_plot, plazos_interp.astype(float))
else:
    tasas_interp = np.interp(plazos_interp, nodos_plot, tasas_plot, left=tasas_plot[0], right=tasas_plot[-1])

plt.figure(figsize=(9, 5))
plt.plot(nodos_plot, tasas_plot * 100, 'o-', label='Nodos de la curva (dato de mercado)')
plt.plot(plazos_interp, tasas_interp * 100, 'x', markersize=10, color='red',
         label='Tasas interpoladas (plazos de nuestros bonos)')
plt.xlabel('Plazo (d\u00edas)')
plt.ylabel('Tasa (%)')
plt.title(f'Curva cup\u00f3n cero disponible ({fecha_curva:%d/%m/%Y}) \u2014 m\u00e9todo: {metodo_interpolacion}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# 💰 Valuación

Ya tenemos parámetros, curvas y funciones de interpolación. Toca aplicar la fórmula de valuación correspondiente a cada instrumento:

| Instrumento | Curva usada | Idea central |
|---|---|---|
| CETES | Cupón cero | Un solo flujo (el nominal) descontado a la tasa del plazo exacto. |
| Bono M (cupón cero) | Cupón cero | Se descuenta **cada cupón por separado**, cada uno a la tasa que le corresponde a su plazo. |
| Bono M (yield) | Yield (TIR) | Se usa **una sola tasa** (la TIR de mercado) y la fórmula cerrada de anualidad de un bono. |
| Bonde D | Cupón cero + spread | Cada cupón es flotante: se calcula con la tasa de fondeo realizada + un spread de mercado, y se descuenta con curva cupón cero + spread. |


## 🧾 CETES o Bonos cupón cero

Un CETE (o cualquier bono cupón cero) promete **un solo pago** al vencimiento: el valor nominal. Su precio es simplemente ese pago, descontado a la tasa correspondiente a su plazo:

$$ V_{CETE} = \frac{N \cdot C}{1 + t_{vp_p} \cdot p/360} $$

Donde:
- $N$: valor nominal del título.
- $C$: número de contratos (la posición; negativo si está en corto).
- $p$: plazo (en días) al vencimiento.
- $t_{vp_p}$: tasa de valor presente (curva cupón cero) interpolada al plazo $p$.


In [ ]:
# @title 🧮 Valuación de CETES { display-mode: "both" }
Xvp_bcc = np.zeros((n, len(plazos_bcc)))

for i in range(n):
    if itpl == 0:
        Xvp_bcc[i, :] = np.interp(plazos_bcc, np.array(nodos_gov[0]), np.array(x_orig.drop('Date', axis=1).iloc[i]))
    else:
        Xvp_bcc[i, :] = talamb(np.array(nodos_gov[0]), np.array(x_orig.drop('Date', axis=1).iloc[i]), plazos_bcc)

# Tasas de valor presente interpoladas para el d\u00eda de valuaci\u00f3n (fila 0)
Valoración_cetes = contratos_bcc * nominal_bcc / (1 + Xvp_bcc[0, :] * plazos_bcc / 360)

resumen_cetes = pd.DataFrame({
    'Plazo (d\u00edas)': plazos_bcc,
    'Contratos': contratos_bcc,
    'Tasa interpolada': np.round(Xvp_bcc[0, :] * 100, 4),
    'Valuaci\u00f3n ($)': np.round(Valoración_cetes, 2),
})
resumen_cetes


# 📄 Bonos M

Un Bono M paga **cupones periódicos** a tasa fija (aquí, cada `plazocupon_bm` días) y regresa el nominal en el último pago. Para valuarlo con el método de cupón cero, primero necesitamos **reconstruir el calendario completo de flujos** de los 5 bonos de ejemplo: en qué día cae cada cupón, cuánto vale, y cuál es el plazo acumulado (desde hoy) a cada uno de esos pagos.

La siguiente celda hace justo eso — es código de "contabilidad de flujos" (bastante denso en índices), por lo que está colapsado como formulario. Lo importante es el resultado: los vectores `VTplazos` (plazo de cada flujo), `contratosT`, `tasafijaT`, etc., todos alineados uno a uno.


In [ ]:
# @title 🗓️ Construcción del calendario de flujos del Bono M { display-mode: "form" }
import numpy as np

# N\u00famero de cupones a pagar por cada uno de los 5 bonos
N = (plazos_bm / plazocupon_bm).astype(int) + 1

# Vectores "aplanados": un elemento por cada flujo (cup\u00f3n) de cada bono
VTplazos = np.zeros((1, int(np.sum(N))))      # plazo acumulado (d\u00edas) de cada flujo
contratosT = np.zeros((1, int(np.sum(N))))    # contratos del bono al que pertenece el flujo
nominalT = np.zeros((1, int(np.sum(N))))      # (no se usa; se deja por compatibilidad)
plazocuponT = np.zeros((1, int(np.sum(N))))   # periodicidad del cup\u00f3n de ese flujo
tasafijaT = np.zeros((1, int(np.sum(N))))     # tasa cup\u00f3n fija de ese flujo
ulNomT = np.zeros((1, int(np.sum(N))))        # nominal que se paga SOLO en el \u00faltimo flujo

# Plazo del primer cup\u00f3n (puede ser "corto" si el bono ya empez\u00f3 a devengar)
plazini = plazos_bm - plazocupon_bm * (N - 1)

for j in range(len(plazos_bm)):
    if j == 0:
        VTplazos[:, 0:int(np.sum(N[0:j+1]))] = np.arange(plazini[j], plazos_bm[j] + plazocupon_bm[j], plazocupon_bm[j])
        contratosT[:, 0:int(np.sum(N[0:j+1]))] = np.arange(contratos_bm[j], contratos_bm[j] + 1)
        plazocuponT[:, 0:int(np.sum(N[0:j+1]))] = plazocupon_bm[j] * np.ones(int(np.sum(N[0:j+1])))
        tasafijaT[:, 0:int(np.sum(N[0:j+1]))] = tfcupon_bm[j] * np.ones(int(np.sum(N[0:j+1])))
        ulNomT[:, N[j]-1] = contratos_bm[j]
    else:
        VTplazos[:, int(np.sum(N[0:j])):int(np.sum(N[0:j+1]))] = np.arange(plazini[j], plazos_bm[j] + plazocupon_bm[j], plazocupon_bm[j])
        contratosT[:, int(np.sum(N[0:j])):int(np.sum(N[0:j+1]))] = np.arange(contratos_bm[j], contratos_bm[j] + 1)
        plazocuponT[:, int(np.sum(N[0:j])):int(np.sum(N[0:j+1]))] = plazocupon_bm[j] * np.ones(int(np.sum(N[j:j+1])))
        tasafijaT[:, int(np.sum(N[0:j])):int(np.sum(N[0:j+1]))] = tfcupon_bm[j] * np.ones(int(np.sum(N[j:j+1])))
        ulNomT[:, int(np.sum(N[0:j+1]))-1] = contratos_bm[j]

# Interpolamos la curva cup\u00f3n cero al plazo de CADA flujo (todos los bonos a la vez)
Xvp = np.zeros((n, VTplazos.shape[1]))
for i in range(n):
    if itpl == 0:
        Xvp[i, :] = np.interp(VTplazos[0, :], np.array(nodos_gov[0]), np.array(x_orig.drop('Date', axis=1).iloc[0]))
    else:
        Xvp[i, :] = talamb(np.array(nodos_gov[0]), np.array(x_orig.drop('Date', axis=1).iloc[0]), VTplazos[0, :])

print("Plazo acumulado de cada flujo (d\u00edas):")
print(VTplazos)
print("\nNominal pagado en cada flujo (solo el \u00faltimo de cada bono es distinto de 0):")
print(ulNomT)


In [ ]:
# @title 🔎 Inspección rápida: plazos calculados { display-mode: "both" }
VTplazos


## 🧮 Fórmula: Bono a tasa fija, descontado con curva cupón cero

$$ V_{cc}=\sum_{i=1}^{n}\frac{N\cdot C \cdot t_{c}\cdot p_c/360}{(1+t_{vp_{p_i}} \cdot p_i/360)} + \frac{N\cdot C}{(1+t_{vp_{p_n}} \cdot p_n/360)}$$

Donde:
- $V_{cc}$: valor del bono bajo la curva cupón cero (precio sucio).
- $N$: valor nominal del bono.
- $C$: número de contratos.
- $p_{c}$: plazo fijo entre cada pago de cupón.
- $p_{i}$: plazo acumulado (en días) al cupón $i$.
- $t_{c}$: tasa cupón fija.
- $t_{vp_{p_i}}$: tasa de valor presente (curva cupón cero) interpolada al plazo acumulado del cupón $i$.

Es decir: **se descuenta cada cupón por separado** a la tasa que le toca según cuándo ocurre, y al final se le suma el nominal descontado al plazo del último cupón. Aquí solo hay un factor de riesgo subyacente: la curva gubernamental.


In [ ]:
# @title 🧮 Función y valuación: Bono M por cupón cero { display-mode: "both" }
import numpy as np

def bonoMcccero(contratosT, nominal, tasafijaT, plazocuponT, VTplazos, Xvp, N):
    """
    Valúa uno o varios bonos a tasa fija descontando cada cup\u00f3n con la
    curva cup\u00f3n cero (ver f\u00f3rmula de la celda anterior).
    Todos los vectores vienen "aplanados" (un elemento por flujo); `N`
    indica cu\u00e1ntos flujos le corresponden a cada bono, en orden.
    """
    V0 = np.zeros((1, len(N)))[0]
    V0f = ((((contratosT * tasafijaT * (plazocuponT / 360)) + ulNomT) / (1 + Xvp * VTplazos / 360)) * nominal)[0]
    for j in range(len(N)):
        if j == 0:
            V0[j] = np.sum(V0f[j:N[j]])
        else:
            V0[j] = np.sum(V0f[sum(N[0:j]):sum(N[0:j+1])])
    return V0

V0 = bonoMcccero(contratosT[0], nominal_bm, tasafijaT[0], plazocuponT[0], VTplazos[0], Xvp[0], N)

resumen_bm_cc0 = pd.DataFrame({
    'Plazo (d\u00edas)': plazos_bm,
    'Contratos': contratos_bm,
    'Tasa cup\u00f3n': tfcupon_bm,
    'Valuaci\u00f3n ($)': np.round(V0, 2),
})
resumen_bm_cc0


## 📄 Bono M — tasa yield

En vez de descontar cada cupón con una tasa distinta, este método usa **una sola tasa** $x$ (la TIR/yield de mercado para ese plazo) y la fórmula cerrada del valor presente de una anualidad más el pago del nominal:

$$ V_{yield} = \left[ N\cdot C\cdot t_c \cdot \frac{p_c}{360}\cdot a(x) + \frac{N\cdot C}{(1+x\cdot p_c/360)^{N}} \right]\cdot (1+x\cdot p_c/360)^{1-\,p_1/p_c} $$

$$ a(x) = \frac{1-(1+x\cdot p_c/360)^{-N}}{p_c\cdot x/360} $$

Donde $N$ es el número de cupones restantes, $p_c$ el plazo fijo de cupón, y $p_1$ el plazo del primer cupón (posiblemente corto). El último factor ajusta el precio al día exacto de valuación cuando ese primer periodo no es completo.

Al usar la misma fecha de valuación y curva, este método y el de cupón cero de la sección anterior deberían dar **resultados muy parecidos** (pequeñas diferencias vienen de que la curva yield y la curva cupón cero no son exactamente la misma cosa).


In [ ]:
# @title 📐 Interpolación de la curva yield { display-mode: "form" }
Xvp_bm_yd = np.zeros((n_yd, len(plazos_bm)))

for i in range(n_yd - 1):
    if itpl == 0:
        Xvp_bm_yd[i, :] = np.interp(plazos_bm, np.array(nodos_gov_yd[0]), np.array(x_orig_yd.drop('Date', axis=1).iloc[i]))
    else:
        Xvp_bm_yd[i, :] = talamb(np.array(nodos_gov_yd[0]), np.array(x_orig_yd.drop('Date', axis=1).iloc[i]), plazos_bm)

print("Tasas yield interpoladas para los plazos de nuestros bonos:")
print(plazos_bm)
Xvp_bm_yd


In [ ]:
# @title 🧮 Función y valuación: Bono M por tasa yield { display-mode: "both" }
import numpy as np

def bonoMyield(x, plazos, plazocupon, tfcupon, nominal, contratos):
    """Valúa un bono a tasa fija descontando con una sola tasa yield `x`."""
    N = (plazos / plazocupon).astype(int) + 1
    a = (1 - (1 + x * plazocupon / 360) ** (-N)) / (plazocupon * x / 360)
    p1 = plazos - plazocupon * (N - 1)
    return ((contratos * nominal * tfcupon * plazocupon / 360) * a + (contratos * nominal) / ((1 + x * plazocupon / 360) ** N)) * (1 + x * plazocupon / 360) ** (1 - p1 / plazocupon)

x0 = Xvp_bm_yd[0, :]  # tasas yield vigentes el d\u00eda de valuaci\u00f3n
Valoracion_bonoM_yield = bonoMyield(x0, plazos_bm, plazocupon_bm, tfcupon_bm, nominal_bm, contratos_bm)

resumen_bm_yield = pd.DataFrame({
    'Plazo (d\u00edas)': plazos_bm,
    'Contratos': contratos_bm,
    'Tasa yield': np.round(x0 * 100, 4),
    'Valuaci\u00f3n ($)': np.round(Valoracion_bonoM_yield, 2),
})
resumen_bm_yield


### 🔍 Comparando ambos métodos

Compara la columna `Valuación ($)` de las dos tablas anteriores: si el método de interpolación, la fecha y las curvas son consistentes entre sí, los resultados de **cupón cero** y **yield** deberían ser prácticamente iguales para el mismo bono. Diferencias grandes suelen indicar que las curvas cupón cero y yield no describen exactamente las mismas condiciones de mercado (son curvas separadas, estimadas de forma independiente).


# 📄 Bondes D

El Bonde D es un bono de **tasa flotante**: en vez de una tasa cupón fija, cada cupón (cada 28 días) se calcula con la **tasa de fondeo bancario realizada** durante ese periodo, más un pequeño ajuste. Esto lo hace un instrumento de bajo riesgo de tasa de interés (su valor casi no cambia cuando cambian las tasas, porque el cupón "se mueve con el mercado").

Para valuarlo necesitamos dos piezas nuevas:

1. **Tasa del cupón en curso** — como el cupón actual ya empezó a devengarse, se combina la tasa de fondeo **ya realizada** (`tf_int`, del inicio del cupón a hoy) con la tasa de fondeo **vigente** (`tf_act`, de hoy al resto del periodo):

$$ t_{cup\acute{o}n} = \left[ \left(1+t_{dev}\cdot\frac{d_{dev}}{360}\right)\cdot\left(1+t_{act}\right)^{\frac{p_c-d_{dev}}{360}\cdot 360} - 1 \right]\cdot\frac{360}{p_c} $$

donde $t_{dev}$ es la tasa geométrica realizada en los $d_{dev}$ días ya transcurridos del cupón, $t_{act}$ es la tasa de fondeo de hoy, y $p_c$ el plazo del cupón (28 días). Los cupones futuros (que aún no empiezan) usan directamente la tasa de fondeo vigente compuesta a 28 días.

2. **Descuento con spread** — el valor presente de cada flujo se calcula con la curva cupón cero **más** una sobretasa/spread de mercado ($t_{st}$), específica para este tipo de instrumento:

$$ V_{f} = \frac{\left(N\cdot C\cdot t_{cup\acute{o}n}\cdot p_c/360\right) + N\cdot C_{\text{(si es el último flujo)}}}{1+(t_{vp_p}+t_{st,p})\cdot p/360} $$

> 🛠️ **Nota de correcciones respecto a la versión original:** al revisar el cuaderno encontramos dos detalles que dejaban la valuación de Bondes D incompleta y los corregimos aquí:
> 1. La rama de interpolación por **tasa alambrada** para las curvas de Bonde D estaba sin implementar (dejaba las tasas en cero). Ahora reutiliza `talamb()`, igual que en CETES y Bono M.
> 2. La función de valuación se estaba llamando con la tasa de fondeo de **hoy** para todos los cupones, en vez de usar el vector `tasafijaT_bd` (que sí distingue el cupón en curso, ya parcialmente devengado, de los cupones futuros). Ahora se usa `tasafijaT_bd`.


In [ ]:
# @title 🗓️ Construcción del calendario de flujos y tasas de Bonde D { display-mode: "form" }
m = len(plazos_bdm)
N_bd = (plazos_bdm // plazocupon_bdm) + 1  # n\u00famero de cupones a pagar

VTplazos_bdm = np.zeros(np.sum(N_bd))
contratos_bdmT = np.zeros(np.sum(N_bd))
plazocupon_bdmT = np.zeros(np.sum(N_bd))
tasafijaT_bd = np.zeros(np.sum(N_bd))
ulNomT_bd = np.zeros(np.sum(N_bd))
Xvp_bd = np.zeros((n, len(VTplazos_bdm)))
Xst_bd = np.zeros((n, len(VTplazos_bdm)))

plazini_bd = plazos_bdm - plazocupon_bdm * (N_bd - 1)
ddv = plazocupon_bdm - plazini_bd  # d\u00edas ya transcurridos del cup\u00f3n en curso
tfcupon = np.zeros(m)
tfcupondev = np.zeros(m)
tfcupgen = ((1 + tf_act / 360) ** (plazocupon_bdm[0]) - 1) * 360 / plazocupon_bdm[0]

for j in range(m):
    tfcupondev[j] = ((np.prod(1 + tf_int[0:ddv[j]] / 360) - 1) * 360) / ddv[j]
    tfcupon[j] = (((1 + tfcupondev[j] * ddv[j] / 360) * (1 + tf_act / 360) ** (plazocupon_bdm[0] - ddv[j]) - 1) * 360) / plazocupon_bdm[0]

print("Tasa realizada durante lo que va del cup\u00f3n en curso:", np.round(tfcupondev, 6))
print("Tasa del cup\u00f3n en curso (realizada + vigente):      ", np.round(tfcupon, 6))

for j in range(m):
    sum_N_bd = np.sum(N_bd[:j+1])
    sum_N_bd_prev = np.sum(N_bd[:j]) if j > 0 else 0

    if j == 0:
        VTplazos_bdm[:sum_N_bd] = np.arange(plazini_bd[j], plazos_bdm[j] + 1, plazocupon_bdm[j])
        contratos_bdmT[:sum_N_bd] = contratos_bdm[j]
        plazocupon_bdmT[:sum_N_bd] = plazocupon_bdm[j]
        ulNomT_bd[sum_N_bd - 1] = contratos_bdm[j]
        tasafijaT_bd[0] = tfcupon[j]
        tasafijaT_bd[1:sum_N_bd] = tfcupgen
    else:
        VTplazos_bdm[sum_N_bd_prev:sum_N_bd] = np.arange(plazini_bd[j], plazos_bdm[j] + 1, plazocupon_bdm[j])
        contratos_bdmT[sum_N_bd_prev:sum_N_bd] = contratos_bdm[j]
        plazocupon_bdmT[sum_N_bd_prev:sum_N_bd] = plazocupon_bdm[j]
        tasafijaT_bd[sum_N_bd_prev] = tfcupon[j]
        tasafijaT_bd[sum_N_bd_prev + 1:sum_N_bd] = tfcupgen
        ulNomT_bd[sum_N_bd - 1] = contratos_bdm[j]

# Interpolamos la curva cup\u00f3n cero y la curva de spread al plazo de cada flujo.
# (Antes, la rama de tasa alambrada (itpl == 1) no estaba implementada; ahora
# reutilizamos talamb(), igual que para CETES y Bono M.)
for i in range(n):
    nodos_arr = np.array(nodos, dtype=float)
    nodos3_arr = np.array(nodos3_bd, dtype=float)
    curva_i = np.array(x_orig.drop('Date', axis=1).iloc[i], dtype=float)
    curva3_i = np.array(X3_orig_bd.drop('Date', axis=1).iloc[i], dtype=float)
    if itpl == 0:
        Xvp_bd[i, :] = np.interp(VTplazos_bdm, nodos_arr, curva_i, left=curva_i[0], right=curva_i[-1])
        Xst_bd[i, :] = np.interp(VTplazos_bdm, nodos3_arr, curva3_i, left=curva3_i[0], right=curva3_i[-1])
    else:
        Xvp_bd[i, :] = talamb(nodos_arr, curva_i, VTplazos_bdm)
        Xst_bd[i, :] = talamb(nodos3_arr, curva3_i, VTplazos_bdm)

X2_pr = X2_orig_bd.sort_values('fecha', ascending=False)


In [ ]:
# @title 🧮 Función y valuación: Bonde D { display-mode: "both" }
def bondeD(contratosT, nominal, tasafijaT, plazocuponT, VTplazos, Xvp, Xst, N, ulNomT):
    """
    Valúa uno o varios bonos de tasa flotante: cada flujo se calcula con su
    propia tasa de cup\u00f3n (`tasafijaT`, ya distingue el cup\u00f3n en curso de
    los futuros) y se descuenta con la curva cup\u00f3n cero m\u00e1s el spread
    (`Xvp + Xst`).
    """
    V0 = np.zeros(len(N))
    V0f = (((contratosT * tasafijaT * (plazocuponT / 360)) + ulNomT) / (1 + (Xvp + Xst) * VTplazos / 360)) * nominal

    for j in range(len(N)):
        if j == 0:
            V0[j] = np.sum(V0f[:N[j]])
        else:
            V0[j] = np.sum(V0f[np.sum(N[:j]):np.sum(N[:j+1])])

    return V0

# Usamos tasafijaT_bd (tasa por cup\u00f3n) en vez de tf_act (tasa \u00fanica de hoy),
# para que cada flujo se valúe con la tasa que realmente le corresponde.
V0_bd = bondeD(contratos_bdmT, nominal_bdm, tasafijaT_bd, plazocupon_bdmT, VTplazos_bdm, Xvp_bd[0, :], Xst_bd[0, :], N_bd, ulNomT_bd)

resumen_bd = pd.DataFrame({
    'Plazo (d\u00edas)': plazos_bdm,
    'Contratos': contratos_bdm,
    'Valuaci\u00f3n ($)': np.round(V0_bd, 2),
})
resumen_bd


# 📊 Resumen comparativo

Juntamos los resultados de los tres instrumentos (cuatro, contando los dos métodos del Bono M) en una sola tabla y una gráfica, para comparar de un vistazo el **valor neto de cada posición**.

Recuerda que cada instrumento usa un conjunto de plazos y contratos de ejemplo distinto — no son la misma cartera — así que la comparación es entre el valor neto total de cada método/instrumento, no bono por bono.


In [ ]:
# @title 📊 Tabla y gráfica de resultados { display-mode: "both" }
from IPython.display import display

def tabla_resumen(nombre, plazos, contratos, valuaciones):
    return pd.DataFrame({
        'Instrumento': nombre,
        'Posici\u00f3n': np.arange(1, len(plazos) + 1),
        'Plazo (d\u00edas)': plazos,
        'Contratos': contratos,
        'Valuaci\u00f3n ($)': np.round(valuaciones, 2),
    })

resumen = pd.concat([
    tabla_resumen('CETES', plazos_bcc, contratos_bcc, Valoración_cetes),
    tabla_resumen('Bono M (cup\u00f3n cero)', plazos_bm, contratos_bm, V0),
    tabla_resumen('Bono M (yield)', plazos_bm, contratos_bm, Valoracion_bonoM_yield),
    tabla_resumen('Bonde D', plazos_bdm, contratos_bdm, V0_bd),
], ignore_index=True)

display(resumen)

orden = ['CETES', 'Bono M (cup\u00f3n cero)', 'Bono M (yield)', 'Bonde D']
totales = resumen.groupby('Instrumento')['Valuaci\u00f3n ($)'].sum().reindex(orden)
print("\nValor neto de la posici\u00f3n por instrumento:")
display(totales.to_frame('Valor neto ($)'))

plt.figure(figsize=(8, 5))
totales.plot(kind='bar', color=['#4C72B0', '#DD8452', '#DD8452', '#55A868'])
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Valor neto de la posici\u00f3n ($)')
plt.title('Comparaci\u00f3n del valor neto por instrumento')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


# ✅ Glosario y para seguir practicando

### Glosario rápido
- **Nominal ($N$):** el monto que se paga al vencimiento de un bono (por unidad, sin contar contratos).
- **Contratos ($C$):** el tamaño de la posición; negativo significa que estamos "cortos" (vendidos).
- **Cupón:** pago periódico de intereses antes del vencimiento.
- **Curva cupón cero:** tasas de descuento puras por plazo, sin cupones de por medio.
- **Curva yield:** la TIR de mercado para un bono cuponado de cierto plazo.
- **Tasa alambrada:** método de interpolación consistente con tasas forward, estándar en el mercado mexicano.
- **Tasa de fondeo:** tasa de referencia de muy corto plazo (overnight) a la que se indexan los cupones de instrumentos de tasa flotante como el Bonde D.
- **Spread/sobretasa:** ajuste adicional sobre la curva base para reflejar riesgo/liquidez propios del instrumento.

### 🧪 Ejercicios sugeridos
1. Cambia `fecha_valuacion` y observa cómo cambia la curva graficada (recuerda la nota sobre qué fecha usa cada archivo).
2. Compara los resultados con `metodo_interpolacion = "Interpolación lineal"` vs. `"Tasa alambrada"`. ¿Qué tan grande es la diferencia?
3. Modifica los `plazos_bm` o `tfcupon_bm` de un bono y recalcula: ¿el precio sube o baja? ¿Tiene sentido con lo que sabes de duración y tasas?
4. Agrega una sexta posición a cualquiera de los instrumentos (edita los campos de formulario para que las listas tengan 6 elementos en vez de 5).
5. ¿Por qué el valor del Bonde D cambia tan poco frente a movimientos de tasas, comparado con el Bono M? Relaciona tu respuesta con el concepto de duración.
